In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đồ họa để hiển thị đẹp hơn
%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
#1. Hàm load_data() và hiển thị 10 dòng đầu tiên
def load_data(file_path):
    # Tải dữ liệu từ file CSV
    df = pd.read_csv(file_path)
    return df

# Gọi hàm load dữ liệu (thay đổi đường dẫn nếu cần)
df = load_data('titanic_disaster.csv')
df.head(10)

In [ ]:
#2. Thống kê và trực quan hóa dữ liệu thiếu bằng Heatmap
# Thống kê số lượng dữ liệu thiếu
missing_data = df.isnull().sum()
print("Thống kê dữ liệu thiếu trên từng biến:")
print(missing_data[missing_data > 0])

# Vẽ biểu đồ Heatmap trực quan dữ liệu thiếu
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Biểu đồ Heatmap vị trí dữ liệu thiếu (Khuyết khuyết)')
plt.show()

In [ ]:
#3. Tách cột Name và xóa cột gốc
# Tách Name dựa trên dấu phẩy đầu tiên giữa Họ và Tên đệm/Danh xưng
# Cấu trúc thông thường: "Braund, Mr. Owen Harris" -> firstName là Họ, secondName là phần còn lại
df['firstName'] = df['Name'].apply(lambda x: x.split(',')[0].strip())
df['secondName'] = df['Name'].apply(lambda x: x.split(',')[1].strip() if ',' in x else x)

# Xóa cột Name gốc
df.drop(columns=['Name'], inplace=True)
df.head(2)

In [ ]:
#4. Rút gọn kích thước dữ liệu cột Sex
# Thay thế male -> M và female -> F
df['Sex'] = df['Sex'].map({'male': 'M', 'female': 'F'})
df.head(2)

In [ ]:
#5. Xử lý dữ liệu thiếu trên biến Age
#a. Vẽ Box plot xác định phân phối tuổi theo hạng vé (Pclass)
plt.figure(figsize=(10, 6))
sns.boxplot(x='Pclass', y='Age', data=df, palette='Set2')
plt.title('Phân phối độ tuổi theo từng Hạng hành khách (Pclass)')
plt.xlabel('Hạng hành khách (Pclass)')
plt.ylabel('Tuổi (Age)')
plt.show()

In [ ]:
#b. Tiến hành thay thế và vẽ lại Heatmap
# Điền tuổi thiếu bằng giá trị trung vị của từng Pclass tương ứng
df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Hiển thị kết quả kiểm tra dạng bảng (ví dụ xem còn null không)
print(f"Số lượng dòng thiếu tuổi sau khi xử lý: {df['Age'].isnull().sum()}")

# Trực quan lại bằng Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Heatmap dữ liệu sau khi đã xử lý thiếu cột Age')
plt.show()

In [ ]:
#6. Xây dựng biến Agegroup (Thang đo thứ tự)
# Định nghĩa các khoảng cắt và nhãn
# Lưu ý: ( age <= 12 ] -> Kid; (12, 18] -> Teen; (18, 60] -> Adult; ( > 60 ) -> Older
bins = [-1, 12, 18, 60, 120]
labels = ['Kid', 'Teen', 'Adult', 'Older']

df['Agegroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
df[['Age', 'Agegroup']].head(5)

In [ ]:
#7. Tách đặc trưng danh xưng (namePrefix)
# Danh xưng thường đứng trước dấu chấm trong cột secondName (Ví dụ: "Mr. Owen Harris")
def extract_prefix(text):
    for prefix in ['Mr', 'Mrs', 'Miss', 'Master']:
        if text.startswith(prefix):
            return prefix
    return 'Other' # Các danh xưng quý tộc hoặc hiếm gặp khác (Dr, Rev, Col,...)

df['namePrefix'] = df['secondName'].apply(extract_prefix)
df[['secondName', 'namePrefix']].head(5)

In [ ]:
#8. Khai thác thông tin familySize
# Công thức: familySize = 1 + SibSp + Parch
# Lưu ý: Yêu cầu ghi là 1 + SibSp + Parch (nếu familySize = 1 tức là đi 1 mình)
df['familySize'] = 1 + df['SibSp'] + df['Parch']
df[['SibSp', 'Parch', 'familySize']].head(5)

In [ ]:
#9. Tạo đặc trưng Alone
# Nếu familySize == 1 tức là đi 1 mình (trong yêu cầu ghi "Nếu familySize = 0 thì Alone = 1", 
# tuy nhiên theo logic chuẩn toán học ở trên familySize tối thiểu là 1. Ta áp dụng theo logic chuẩn: nếu đi 1 mình thì Alone = 1)
df['Alone'] = df['familySize'].apply(lambda x: 1 if x == 1 else 0)
df[['familySize', 'Alone']].head(5)

In [ ]:
#10. Tách loại cabin (typeCabin)
# Lấy chữ cái đầu tiên của Cabin, nếu thiếu điền "Unknown"
df['Cabin'] = df['Cabin'].fillna('Unknown')
df['typeCabin'] = df['Cabin'].apply(lambda x: x[0] if x != 'Unknown' else 'Unknown')
df[['Cabin', 'typeCabin']].head(5)

In [ ]:
#11. Loại bỏ dữ liệu trùng lặp giữa tập Train và Test
# Loại bỏ các hàng trùng lặp hoàn toàn (nếu có), ưu tiên giữ lại hàng xuất hiện trước
df.drop_duplicates(keep='first', inplace=True)
print(f"Kích thước tập dữ liệu hiện tại: {df.shape}")

In [ ]:
#12. Trực quan tương quan tỉ lệ sống sót trên từng nhóm giới tính
plt.figure(figsize=(6, 5))
sns.countplot(x='Sex', hue='Survived', data=df, palette='Set1')
plt.title('Tỷ lệ sống sót và thiệt mạng theo Giới tính')
plt.xlabel('Giới tính (M: Nam, F: Nữ)')
plt.ylabel('Số lượng hành khách')
plt.legend(['Thiệt mạng (0)', 'Sống sót (1)'])
plt.show()

In [ ]:
#13. Trực quan thông tin hành khách sống sót trên từng nhóm phân loại (Pclass)
plt.figure(figsize=(7, 5))
sns.barplot(x='Pclass', y='Survived', data=df, palette='muted', errorbar=None)
plt.title('Tỷ lệ sống sót trung bình theo Hạng vé (Pclass)')
plt.xlabel('Hạng vé')
plt.ylabel('Tỷ lệ sống sót')
plt.show()

In [ ]:
#14. Trực quan thông tin sống sót theo giới tính và thang đo tuổi tác (Agegroup)
plt.figure(figsize=(10, 6))
sns.barplot(x='Agegroup', y='Survived', hue='Sex', data=df, palette='pastel', errorbar=None)
plt.title('Tỷ lệ sống sót theo Nhóm tuổi và Giới tính')
plt.xlabel('Nhóm tuổi')
plt.ylabel('Tỷ lệ sống sót')
plt.show()

In [ ]:
#15. Trực quan xác suất sống sót dựa trên thông tin nhóm đi cùng (familySize / Alone)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Theo familySize
sns.barplot(ax=axes[0], x='familySize', y='Survived', data=df, palette='Blues', errorbar=None)
axes[0].set_title('Tỷ lệ sống sót theo Số lượng thành viên gia đình')

# Theo Alone
sns.barplot(ax=axes[1], x='Alone', y='Survived', data=df, palette='Oranges', errorbar=None)
axes[1].set_title('Tỷ lệ sống sót: Đi một mình (1) vs Đi theo nhóm (0)')

plt.show()

In [ ]:
#16. Trực quan xác suất hành khách sống sót dựa trên thông tin giá vé (Fare)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df[df['Survived'] == 1], x='Fare', label='Sống sót', fill=True, color='green', alpha=0.5)
sns.kdeplot(data=df[df['Survived'] == 0], x='Fare', label='Thiệt mạng', fill=True, color='red', alpha=0.5)
plt.title('Phân phối mật độ giá vé (Fare) theo trạng thái Sống/Chết')
plt.xlabel('Giá vé (Fare)')
plt.xlim(0, 200) # Giới hạn biên để dễ quan sát vùng tập trung dữ liệu lớn
plt.legend()
plt.show()

In [ ]:
#17. Trực quan số lượng người thiệt mạng và sống sót theo Pclass và Cảng cập bến (Embarked)
g = sns.FacetGrid(df, col="Embarked", height=4, aspect=1.2)
g.map(sns.countplot, "Pclass", hue="Survived", data=df, palette="Set2", order=[1, 2, 3], hue_order=[0, 1])
g.add_legend(title="Survived")
g.set_axis_labels("Hạng hành khách (Pclass)", "Số lượng người")
plt.subplots_adjust(top=0.8)
g.fig.suptitle('Số lượng sống/chết theo Hạng hành khách tách theo từng Cảng lên tàu (Embarked)')
plt.show()